# Per-location air-quality baselines
Run from this directory to refit all baselines. The [current per-run results](baseline_results.csv) and [five-model comparison](results/integration/CURRENT_RESULTS.md) include the lag and time features.

In [ ]:
import pandas as pd 
import numpy as np 

P = "processed"
X_tr = pd.read_csv(f"{P}/X_global_train.csv")
X_te = pd.read_csv(f"{P}/X_global_test.csv")
Y_tr = pd.read_csv(f"{P}/Y_global_train.csv")
Y_te = pd.read_csv(f"{P}/Y_global_test.csv")
info_tr = pd.read_csv(f"{P}/global_train_info.csv")
info_te = pd.read_csv(f"{P}/global_test_info.csv")

print(X_tr.shape, Y_tr.shape, info_tr.shape)
print(info_tr["City"].value_counts())

In [ ]:
city = "London"

m_tr = info_tr["City"] == city 
m_te = info_te["City"] == city

X_tr_c, Y_tr_c = X_tr[m_tr], Y_tr[m_tr]
X_te_c, Y_te_c = X_te[m_te], Y_te[m_te]

print(X_tr_c.shape, Y_tr_c.shape)
print(X_te_c.shape, Y_te_c.shape)

In [ ]:
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

target = "PM25_1hr_ahead"
y_tr = Y_tr_c[target]
y_te = Y_te_c[target]

def score(y_true, y_pred):
    return {"r2": r2_score(y_true, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
            "mae": mean_absolute_error(y_true, y_pred)}
    
lr = LinearRegression().fit(X_tr_c, y_tr)
print("LR:         ", score(y_te, lr.predict(X_te_c)))

# persistence: "PM2.5 an hour from now = PM2.5 right now"
print("Persistence:", score(y_te, X_te_c["PM2.5"]))


In [ ]:
print(dict(zip(X_tr_c.columns, lr.coef_.round(3))))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import time

In [ ]:
HORIZONS = [1, 6, 24]
DATASETS = {"global": "City", "beijing": "station"}

def load(name):
    P = "processed"
    return (pd.read_csv(f"{P}/X_{name}_train.csv"), pd.read_csv(f"{P}/X_{name}_test.csv"),
            pd.read_csv(f"{P}/Y_{name}_train.csv"), pd.read_csv(f"{P}/Y_{name}_test.csv"),
            pd.read_csv(f"{P}/{name}_train_info.csv"), pd.read_csv(f"{P}/{name}_test_info.csv"))

rows = []
t0 = time.time()
for name, loc_col in DATASETS.items():
    X_tr, X_te, Y_tr, Y_te, info_tr, info_te = load(name)
    for loc in sorted(info_tr[loc_col].unique()):
        m_tr, m_te = info_tr[loc_col] == loc, info_te[loc_col] == loc
        Xtr, Xte = X_tr[m_tr], X_te[m_te]
        for h in HORIZONS:
            t = f"PM25_{h}hr_ahead"
            ytr, yte = Y_tr.loc[m_tr, t], Y_te.loc[m_te, t]

            SEEDS = [0, 1, 2, 3, 4]
            preds = {
                ("Mean", 0): np.full(len(yte), ytr.mean()),
                ("Persistence", 0): Xte["PM2.5"].values,
                ("Linear Regression", 0): LinearRegression().fit(Xtr, ytr).predict(Xte),
            }
            for s in SEEDS:
                rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                                           max_features=0.5, n_jobs=-1, random_state=s)
                preds[("Random Forest", s)] = rf.fit(Xtr, ytr).predict(Xte)

            for (model, seed), p in preds.items():
                rows.append({"dataset": name, "location": loc, "horizon": h,
                             "model": model, "seed": seed, **score(yte, p)})
        print(f"[{name}] {loc:14s} done ({time.time() - t0:.0f}s)")

results = pd.DataFrame(rows)
results.to_csv("baseline_results.csv", index=False)

In [ ]:
summary = (results.groupby(["dataset", "horizon", "model"])[["r2", "rmse", "mae"]]
                  .mean().round(3))
summary